### Import

In [57]:
import os
import cv2
import dlib

import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from tqdm import tqdm

from joblib import Parallel, delayed
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
from pytesseract import image_to_string
from tensorflow.keras.utils import load_img, img_to_array

### Constants

In [81]:
POSTER_FOLDER = 'dataset/poster/'

# Specify cores to use
# -1 for all cores
N_JOBS = -1

# Batch size
BATCH_SIZE = 100

### CSV check

In [59]:
# Load the data
train_df = pd.read_csv('dataset/dataframe.csv')

In [60]:
# Check for null values
print(train_df.isnull().sum())

ID                     0
Movie_Name             0
Movie_Rating           0
Movie_Certificate    159
Movie_Year             0
Movie_Runtime          0
Movie_Cast             0
Movie_Genre            0
Movie_Poster           0
Movie_Description      0
Movie_Poster_HD        0
All_Movie_Info         0
dtype: int64


In [61]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')

    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        # Apply KPI extraction functions on the image
        # Append KPIs to the dataframe
        # ...
    else:
        print("Error on file " + 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [00:15<00:00, 62.90it/s]


> A lot of columns have missing values, we will adapt our code accordingly

In [62]:
# Split the Genre column into a list of genres
# imdb_movies_df['genre'] = imdb_movies_df['genre'].str.split(',')

# Encode genres into separate columns
# imdb_movies_df = imdb_movies_df.join(imdb_movies_df.pop('genre').str.get_dummies(','))

In [63]:
model = VGG16(weights='imagenet', include_top=False)

### Utils

In [64]:
def process_kpis(movie_id, img_folder, image_funcs):
    image_path = os.path.join(img_folder, str(movie_id) + '.jpg')

    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        # Apply the provided image processing function
        results = [func(img) for func in image_funcs]

        return movie_id, results
    else:
        print("Error: file does not exist", image_path)

        return movie_id, [None for _ in image_funcs]

In [65]:
def update_dataframe(df, results, columns):
    for movie_id, result_list in results:
        for column, result in zip(columns, result_list):
            df.at[movie_id, column] = result


### KPI functions

#### Colors

In [66]:
def avg_rgb_intensity(image):
    average_color_per_row = np.average(image, axis=0)
    average_color = np.average(average_color_per_row, axis=0)

    return average_color

In [90]:
from sklearn.cluster import KMeans

def dominant_color(image):
    # Reshape the image to be a list of pixels
    pixels = image.reshape(-1, 3)

    # Fit a KMeans clustering model to the data
    kmeans = KMeans(n_clusters=5, n_init=10)
    kmeans.fit(pixels)

    # Count the number of pixels assigned to each cluster
    counts = np.bincount(kmeans.labels_)

    # The dominant color is the mean color of the most common cluster
    dominant = kmeans.cluster_centers_[counts.argmax()]

    colors = {
        "Red": [255, 0, 0],
        "Green": [0, 255, 0],
        "Blue": [0, 0, 255],
        "White": [255, 255, 255],
        "Black": [0, 0, 0]
    }

    return min(colors.keys(), key=lambda color: np.linalg.norm(dominant-colors[color]))

In [68]:
def num_distinct_colors(image):
    return len(set(tuple(v) for m2d in image for v in m2d))

#### Faces

In [69]:
def has_face_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)

    return len(faces) > 0


In [70]:
def num_faces_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)

    return len(faces)

In [71]:
def has_face_center(image):
    height, width, _ = image.shape
    center = image[height//4:3*height//4, width//4:3*width//4]
    faces = has_face_dlib(center)

    return faces

#### Text

In [72]:
def has_text_top_quarter(image):
    height, _, _ = image.shape
    top_quarter = image[:height//4, :]
    text = image_to_string(top_quarter)

    return 0 if text.strip() == "" else 1

In [73]:
def has_text_bottom_half(image):
    height, _, _ = image.shape
    bottom_half = image[height//2:, :]
    text = image_to_string(bottom_half)

    return 0 if text.strip() == "" else 1

In [74]:
def num_text_lines_bottom_quarter(image):
    height, _, _ = image.shape
    bottom_quarter = image[3*height//4:, :]
    text = image_to_string(bottom_quarter)
    lines = text.split("\n")

    return len(lines)

In [75]:
from pytesseract import pytesseract, Output

def largest_text_percentage(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    max_area = 0

    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
        max_area = max(max_area, w * h)

    # return as percentage
    return (max_area * 100) / (image.shape[0] * image.shape[1])

In [76]:
def is_text_in_bottom_quarter(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])

    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])

        # If the center of the bounding box is in the bottom quarter
        if y + h/2 > image.shape[0] * 0.75:
            return 1

    return 0

### Process kpi extraction

In [82]:
import warnings


warnings.filterwarnings("ignore")

In [77]:
batches = [train_df['ID'][i:i + BATCH_SIZE] for i in range(0, len(train_df['ID']), BATCH_SIZE)]

# use delayed function from joblib to process images in parallel
parallel_kpi_func_launch = lambda batch, funcs: Parallel(n_jobs=N_JOBS)(delayed(process_kpis)(movie_id, POSTER_FOLDER, funcs) for movie_id in batch)

In [83]:
for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, [avg_rgb_intensity])

    # Hand by hand update of the dataframe cause avg_colors is a list of lists
    for movie_id, avg_color in results:
        if avg_color is not None:
            train_df.at[movie_id, 'avg_red'] = avg_color[0][0]
            train_df.at[movie_id, 'avg_green'] = avg_color[0][1]
            train_df.at[movie_id, 'avg_blue'] = avg_color[0][2]

100%|██████████| 10/10 [00:05<00:00,  1.84it/s]


In [79]:
funcs, columns = zip(*[
    (has_face_dlib, 'has_face'),
    (num_faces_dlib, 'num_faces'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)


 10%|█         | 1/10 [00:18<02:44, 18.30s/it]/Users/hparraud/Dev/Github/Nostradamovies/data/venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 10/10 [02:53<00:00, 17.35s/it]


In [84]:
funcs, columns = zip(*[
    (has_text_top_quarter, 'has_text_top_quarter'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [01:16<00:00,  7.64s/it]


In [85]:
funcs, columns = zip(*[
    (has_face_center, 'has_face_center'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [01:03<00:00,  6.40s/it]


In [86]:
funcs, columns = zip(*[
    (has_text_bottom_half, 'has_text_bottom_half'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [02:25<00:00, 14.56s/it]


In [87]:
funcs, columns = zip(*[
    (num_text_lines_bottom_quarter, 'num_text_lines_bottom_quarter'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [01:36<00:00,  9.60s/it]


In [88]:
funcs, columns = zip(*[
    (largest_text_percentage, 'largest_text_percentage'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [04:04<00:00, 24.48s/it]


In [91]:
funcs, columns = zip(*[
    (dominant_color, 'dominant_color'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [34:22<00:00, 206.27s/it]


In [92]:
funcs, columns = zip(*[
    (is_text_in_bottom_quarter, 'is_text_in_bottom_quarter'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [03:57<00:00, 23.75s/it]


In [93]:
funcs, columns = zip(*[
    (num_distinct_colors, 'num_distinct_colors'),
])

for batch in tqdm(batches):
    results = parallel_kpi_func_launch(batch, funcs)
    update_dataframe(train_df, results, columns)

100%|██████████| 10/10 [05:01<00:00, 30.13s/it]


### Export train csv

In [94]:
train_df.to_csv("test.csv")

---

### PAS MIS A JOUR


In [ ]:
def is_title_in_image(image, title):
    extracted_text = pytesseract.image_to_string(image)
    return int(title in extracted_text)

# Add this feature to the dataframe
train_df['title_in_image'] = train_df.apply(lambda row: is_title_in_image(cv2.imread(f'dataset/poster/{row["ID"]}.jpg'), row["Movie_Name"]), axis=1)

In [ ]:
def is_year_in_image(image, year):
    extracted_text = pytesseract.image_to_string(image)
    return int(str(year) in extracted_text)

# Add this feature to the dataframe
train_df['year_in_image'] = train_df.apply(lambda row: is_year_in_image(cv2.imread(f'dataset/poster/{row["ID"]}.jpg'), row["Movie_Year"]), axis=1)

In [ ]:
from tqdm import tqdm

tqdm.pandas()

def is_rating_in_image(image, rating):
    extracted_text = pytesseract.image_to_string(image)
    return int(str(rating) in extracted_text)

# Add this feature to the dataframe
train_df['rating_in_image'] = train_df.progress_apply(lambda row: is_rating_in_image(cv2.imread(f'dataset/poster/{row["ID"]}.jpg'), row["Movie_Rating"]), axis=1)

In [ ]:
import pytesseract
from pytesseract import Output

def largest_text_area_ratio(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    areas = [d['width'][i] * d['height'][i] for i in range(n_boxes)]
    largest_area = max(areas) if areas else 0
    total_area = image.shape[0] * image.shape[1]
    return largest_area / total_area

train_df['largest_text_area_ratio'] = train_df['ID'].progress_apply(lambda id: largest_text_area_ratio(cv2.imread(f'dataset/poster/{id}.jpg')))


In [ ]:
import dlib

def faces_area_ratio(image):
    face_detector = dlib.get_frontal_face_detector()
    faces = face_detector(image, 1)
    total_faces_area = sum([rect.width() * rect.height() for rect in faces])
    total_image_area = image.shape[0] * image.shape[1]
    return total_faces_area / total_image_area

train_df['faces_area_ratio'] = train_df['ID'].progress_apply(lambda id: faces_area_ratio(cv2.imread(f'dataset/poster/{id}.jpg')))

In [ ]:
def has_central_text(image):
    central_region = image[int(image.shape[0] * 0.25):int(image.shape[0] * 0.75), int(image.shape[1] * 0.25):int(image.shape[1] * 0.75)]
    d = pytesseract.image_to_data(central_region, output_type=Output.DICT)
    return int(len(d['text']) > 0)

train_df['has_central_text'] = train_df['ID'].progress_apply(lambda id: has_central_text(cv2.imread(f'dataset/poster/{id}.jpg')))

In [ ]:
def std_dev_color_intensities(image):
    # calculate standard deviation for each color channel
    std_dev_red = np.std(image[:,:,0])
    std_dev_green = np.std(image[:,:,1])
    std_dev_blue = np.std(image[:,:,2])
    return std_dev_red, std_dev_green, std_dev_blue

train_df[['std_dev_red', 'std_dev_green', 'std_dev_blue']] = train_df['ID'].progress_apply(lambda id: pd.Series(std_dev_color_intensities(cv2.imread(f'dataset/poster/{id}.jpg'))))

In [ ]:
def color_intensity_peaks(image):
    # calculate histogram and find peak for each color channel
    hist_red = cv2.calcHist([image],[0],None,[256],[0,256])
    hist_green = cv2.calcHist([image],[1],None,[256],[0,256])
    hist_blue = cv2.calcHist([image],[2],None,[256],[0,256])
    return np.argmax(hist_red), np.argmax(hist_green), np.argmax(hist_blue)

train_df[['peak_red', 'peak_green', 'peak_blue']] = train_df['ID'].progress_apply(lambda id: pd.Series(color_intensity_peaks(cv2.imread(f'dataset/poster/{id}.jpg'))))

In [ ]:
def has_warm_colors(image):
    # define range for warm colors
    lower = np.array([0, 70, 50])
    upper = np.array([35, 255, 255])
    # convert image to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # create a mask for warm colors
    mask = cv2.inRange(hsv, lower, upper)
    return int(np.any(mask))

train_df['has_warm_colors'] = train_df['ID'].progress_apply(lambda id: has_warm_colors(cv2.imread(f'dataset/poster/{id}.jpg')))

In [ ]:
def has_cool_colors(image):
    # define range for cool colors
    lower = np.array([60, 70, 50])
    upper = np.array([160, 255, 255])
    # convert image to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # create a mask for cool colors
    mask = cv2.inRange(hsv, lower, upper)
    return int(np.any(mask))

train_df['has_cool_colors'] = train_df['ID'].progress_apply(lambda id: has_cool_colors(cv2.imread(f'dataset/poster/{id}.jpg')))

In [ ]:
def calculate_aspect_ratio(image):
    # Get image dimensions
    height, width, _ = image.shape
    # Calculate aspect ratio
    aspect_ratio = width / height
    return aspect_ratio

train_df['aspect_ratio'] = train_df['ID'].progress_apply(lambda id: calculate_aspect_ratio(cv2.imread(f'dataset/poster/{id}.jpg')))
